# Cortex Search: Embeddings, Indexes & Hybrid Search

This notebook demonstrates the latest Cortex Search capabilities, including features from the **March 12, 2026 GA release** for multi-index search and custom vector embeddings.

**Topics covered:**
1. **Reusable Embeddings** — Generate embeddings with `AI_EMBED`, store in `VECTOR` columns, reuse for similarity search and Cortex Search
2. **Embeddings & Indexes Co-located with Data** — Managed embeddings (`ON` clause), multi-index services (`TEXT INDEXES` / `VECTOR INDEXES`), hybrid scoring
3. **Reusing Indexes in Custom Apps** — Python API (`snowflake.core.Root`), `SEARCH_PREVIEW` SQL function
4. **Programmatic Access / REST API** — `curl` examples for Cortex Search and Cortex Embed REST endpoints with PAT auth

## Setup

In [ ]:

USE DATABASE SE_FEATURES;
CREATE SCHEMA IF NOT EXISTS SE_FEATURES.CORTEX_SEARCH;
USE SCHEMA SE_FEATURES.CORTEX_SEARCH;
USE WAREHOUSE WHS;

---
## 1. Reusable Embeddings

Embeddings are dense vector representations of text. Once generated, they can be **stored** in a `VECTOR` column and **reused** for:
- Direct similarity search via `VECTOR_COSINE_SIMILARITY`
- As pre-computed input to a Cortex Search service (avoiding re-embedding at index build time)

We use `AI_EMBED` with the `snowflake-arctic-embed-l-v2.0` model, which produces 1024-dimensional vectors.

### 1a. Create sample data and generate embeddings

In [ ]:
CREATE OR REPLACE TABLE support_tickets (
    ticket_id INT,
    title VARCHAR,
    description VARCHAR,
    category VARCHAR,
    priority INT,
    created_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    description_embedding VECTOR(FLOAT, 1024)
);

In [ ]:
INSERT INTO support_tickets (ticket_id, title, description, category, priority, created_at)
VALUES
  (1, 'Login failure after password reset',
   'After resetting my password via the email link, I still cannot log in. The system says invalid credentials even though I just changed it.',
   'Authentication', 1, '2026-03-10 09:15:00'),
  (2, 'Dashboard loading slowly',
   'The analytics dashboard takes over 30 seconds to load. All other pages work fine. Started happening after the last update.',
   'Performance', 2, '2026-03-11 14:30:00'),
  (3, 'Cannot export reports to PDF',
   'When I click Export to PDF, the spinner runs for a minute and then shows a timeout error. CSV export works fine.',
   'Export', 2, '2026-03-12 10:00:00'),
  (4, 'Two-factor authentication not sending SMS',
   'I enabled 2FA on my account but never receive the SMS verification code. Email codes work, but SMS does not arrive.',
   'Authentication', 1, '2026-03-12 11:45:00'),
  (5, 'Data sync between mobile and web out of date',
   'Changes I make on the mobile app do not reflect on the web version for hours. Real-time sync seems broken.',
   'Data Sync', 3, '2026-03-13 08:20:00'),
  (6, 'API rate limiting too aggressive',
   'Our integration is hitting rate limits at only 50 requests per minute. The documented limit is 200 req/min.',
   'API', 2, '2026-03-13 16:00:00'),
  (7, 'Billing page shows incorrect charges',
   'My March invoice shows charges for a Pro plan, but I downgraded to Basic last month. The amount is $200 higher than expected.',
   'Billing', 1, '2026-03-14 07:00:00'),
  (8, 'Search returns irrelevant results',
   'Searching for password reset in the help center returns articles about billing. The relevance seems completely off.',
   'Search', 3, '2026-03-14 09:30:00');

In [ ]:
UPDATE support_tickets
SET description_embedding = AI_EMBED('snowflake-arctic-embed-l-v2.0', description);

In [ ]:
SELECT ticket_id, title,
       SUBSTRING(description_embedding::VARCHAR, 1, 80) AS embedding_preview
FROM support_tickets
LIMIT 3;

### 1b. Similarity search with VECTOR_COSINE_SIMILARITY

The stored embeddings can be used directly for similarity search — no Cortex Search service required. This is useful for ad-hoc similarity queries or when you need full control over the ranking logic.

In [ ]:
SELECT ticket_id, title, category,
       VECTOR_COSINE_SIMILARITY(
           description_embedding,
           AI_EMBED('snowflake-arctic-embed-l-v2.0', 'I can''t log in to my account')
       ) AS similarity
FROM support_tickets
ORDER BY similarity DESC
LIMIT 5;

### 1c. Reuse the same embeddings in a Cortex Search service

The same pre-computed `description_embedding` column can feed a Cortex Search service via the `VECTOR INDEXES` clause — no need to re-embed at service creation time. This is shown in Section 2b.

---
## 2. Embeddings & Indexes Co-located with Data

Cortex Search supports two service creation patterns:

| Pattern | Syntax | Embedding | Use case |
|---|---|---|---|
| **Managed embeddings** | `ON column` | Snowflake generates & manages embeddings | Simple setup, single search column |
| **Multi-index** | `TEXT INDEXES` + `VECTOR INDEXES` | User-provided and/or managed | Multiple searchable columns, hybrid scoring, custom embeddings |

Both patterns became **Generally Available on March 12, 2026**.

### 2a. Managed embeddings service (ON clause)

The simplest approach — Snowflake handles embedding generation and index management.

In [ ]:
CREATE OR REPLACE CORTEX SEARCH SERVICE ticket_search_managed
  ON description
  ATTRIBUTES category, priority
  WAREHOUSE = WHS
  TARGET_LAG = '1 hour'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
AS (
  SELECT ticket_id, title, description, category, priority, created_at
  FROM support_tickets
);

### 2b. Multi-index service with TEXT INDEXES and VECTOR INDEXES

This service indexes **multiple columns** with different index types:
- `TEXT INDEXES` — keyword/lexical search on `title` and `description`
- `VECTOR INDEXES` — managed vector embeddings on `description`, plus the **user-provided** `description_embedding` column (with `query_model` for automatic text-to-vector at query time)

This enables fine-grained hybrid search where you control which indexes are queried and how results are scored.

In [ ]:
CREATE OR REPLACE CORTEX SEARCH SERVICE ticket_search_multi_index
  TEXT INDEXES title, description
  VECTOR INDEXES
    description (model='snowflake-arctic-embed-l-v2.0'),
    description_embedding (query_model='snowflake-arctic-embed-l-v2.0')
  ATTRIBUTES category, priority
  WAREHOUSE = WHS
  TARGET_LAG = '1 hour'
AS (
  SELECT ticket_id, title, description, category, priority, created_at,
         description_embedding
  FROM support_tickets
);

### 2c. Hybrid search with scoring_config

Multi-index services support rich `scoring_config` to control result ranking:
- **`weights`** — balance text, vector, and reranker components
- **`numeric_boosts`** — boost results by a numeric column (e.g., priority)
- **`time_decays`** — prefer more recent documents

In [ ]:
SELECT PARSE_JSON(
  SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    'SE_FEATURES.CORTEX_SEARCH.ticket_search_multi_index',
    '{
      "query": "authentication login issues",
      "columns": ["ticket_id", "title", "category", "priority", "created_at"],
      "filter": {"@eq": {"category": "Authentication"}},
      "limit": 5,
      "scoring_config": {
        "weights": {
          "texts": 2,
          "vectors": 3,
          "reranker": 1
        },
        "functions": {
          "numeric_boosts": [
            {"column": "priority", "weight": 1.5}
          ],
          "time_decays": [
            {"column": "created_at", "weight": 1.0, "limit_hours": 168}
          ]
        }
      }
    }'
  )
)['results'] AS results;

---
## 3. Reusing Indexes in Custom Apps

Once a Cortex Search service is created, it exposes a low-latency query API. You can query it from:
- **Python API** — `snowflake.core.Root` (recommended for applications)
- **SQL** — `SNOWFLAKE.CORTEX.SEARCH_PREVIEW` (for interactive testing)
- **REST API** — direct HTTP calls (covered in Section 4)

### 3a. Python API — simple search with filters and column selection

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.core import Root

session = get_active_session()
root = Root(session)

svc = (
    root
    .databases["SE_FEATURES"]
    .schemas["CORTEX_SEARCH"]
    .cortex_search_services["ticket_search_managed"]
)

resp = svc.search(
    query="export timeout error",
    columns=["ticket_id", "title", "description", "category"],
    filter={"@eq": {"category": "Export"}},
    limit=3,
)

print("Python API — filtered search results:\n")
for r in resp.results:
    print(f"  [{r['ticket_id']}] {r['title']}")
    print(f"    {r['description'][:100]}...\n")

### 3b. Python API — multi_index_query with scoring_config

Target specific indexes and control scoring weights. Here we query only the `description` text index and the `description_embedding` vector index.

In [ ]:
multi_svc = (
    root
    .databases["SE_FEATURES"]
    .schemas["CORTEX_SEARCH"]
    .cortex_search_services["ticket_search_multi_index"]
)

resp = multi_svc.search(
    multi_index_query={
        "description": [{"text": "slow performance dashboard"}],
        "description_embedding": [{"text": "slow performance dashboard"}],
    },
    scoring_config={
        "weights": {
            "texts": 2,
            "vectors": 3,
            "reranker": 1,
        },
        "functions": {
            "vector_boosts": [
                {"weight": 2.0, "column": "description_embedding"},
            ],
            "text_boosts": [
                {"weight": 1.5, "column": "description"},
            ],
        },
    },
    columns=["ticket_id", "title", "category", "description"],
    limit=5,
)

print("Python API — multi-index query results:\n")
for r in resp.results:
    print(f"  [{r['ticket_id']}] {r['title']} ({r['category']})")

### 3c. SQL — SEARCH_PREVIEW function

`SEARCH_PREVIEW` is ideal for interactive testing in worksheets and notebooks. It operates on string literals and is **not** intended for production serving (use the Python or REST API instead).

In [ ]:
SELECT PARSE_JSON(
  SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    'SE_FEATURES.CORTEX_SEARCH.ticket_search_managed',
    '{
      "query": "billing overcharged wrong amount",
      "columns": ["ticket_id", "title", "category"],
      "limit": 3
    }'
  )
)['results'] AS results;

In [ ]:
SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
  'SE_FEATURES.CORTEX_SEARCH.ticket_search_multi_index',
  '{
    "multi_index_query": {
      "title": [{"text": "export PDF"}],
      "description": [{"text": "export timeout"}]
    },
    "columns": ["ticket_id", "title", "description"],
    "limit": 3,
    "scoring_config": {
      "weights": {"texts": 3, "vectors": 1, "reranker": 1},
      "functions": {
        "text_boosts": [{"weight": 2.0, "column": "title"}]
      }
    }
  }'
) AS results;

---
## 4. Programmatic Access / REST API

Both Cortex Search and Cortex Embed expose REST endpoints that can be called from any language or tool. Authentication uses **Programmatic Access Tokens (PATs)**.

> **Note:** The `curl` examples below are templates — replace `<ACCOUNT_URL>` and `<PAT>` with your values.

### 4a. Cortex Search REST API

**Endpoint pattern:**
```
POST https://<ACCOUNT_URL>/api/v2/databases/<DB>/schemas/<SCHEMA>/cortex-search-services/<SERVICE_NAME>:query
```

**curl example — simple query:**
```bash
curl --location "https://<ACCOUNT_URL>/api/v2/databases/CORTEX_SEARCH_DEMO/schemas/SEARCH_LAB/cortex-search-services/TICKET_SEARCH_MANAGED:query" \
  --header 'Content-Type: application/json' \
  --header 'Accept: application/json' \
  --header 'X-Snowflake-Authorization-Token-Type: PROGRAMMATIC_ACCESS_TOKEN' \
  --header "Authorization: Bearer $PAT" \
  --data '{
    "query": "login authentication issues",
    "columns": ["ticket_id", "title", "category"],
    "filter": {"@eq": {"category": "Authentication"}},
    "limit": 5
  }'
```

**curl example — multi-index with scoring_config:**
```bash
curl --location "https://<ACCOUNT_URL>/api/v2/databases/CORTEX_SEARCH_DEMO/schemas/SEARCH_LAB/cortex-search-services/TICKET_SEARCH_MULTI_INDEX:query" \
  --header 'Content-Type: application/json' \
  --header 'Accept: application/json' \
  --header 'X-Snowflake-Authorization-Token-Type: PROGRAMMATIC_ACCESS_TOKEN' \
  --header "Authorization: Bearer $PAT" \
  --data '{
    "query": "slow dashboard",
    "columns": ["ticket_id", "title", "category"],
    "limit": 5,
    "scoring_config": {
      "weights": {"texts": 2, "vectors": 3, "reranker": 1},
      "functions": {
        "numeric_boosts": [{"column": "priority", "weight": 1.5}]
      }
    }
  }'
```

### 4b. Cortex Embed REST API

Generate embeddings over HTTP without a Snowflake session — useful for external applications that need to pre-compute vectors or build vector queries.

**Endpoint pattern:**
```
POST https://<ACCOUNT_URL>/api/v2/cortex/inference:embed
```

**curl example:**
```bash
curl --location "https://<ACCOUNT_URL>/api/v2/cortex/inference:embed" \
  --header 'Content-Type: application/json' \
  --header 'Accept: application/json' \
  --header 'X-Snowflake-Authorization-Token-Type: PROGRAMMATIC_ACCESS_TOKEN' \
  --header "Authorization: Bearer $PAT" \
  --data '{
    "model": "snowflake-arctic-embed-l-v2.0",
    "text": [
      "login failure after password reset",
      "dashboard performance issues"
    ]
  }'
```

**Response shape:**
```json
{
  "object": "list",
  "data": [
    {"object": "embedding", "embedding": [[0.012, -0.034, ...]], "index": 0},
    {"object": "embedding", "embedding": [[-0.021, 0.045, ...]], "index": 1}
  ],
  "model": "snowflake-arctic-embed-l-v2.0",
  "usage": {"total_tokens": 14}
}
```

---
## Cleanup

Drop all resources created in this notebook.

In [ ]:
DROP CORTEX SEARCH SERVICE IF EXISTS SE_FEATURES.CORTEX_SEARCH.ticket_search_managed;
DROP CORTEX SEARCH SERVICE IF EXISTS SE_FEATURES.CORTEX_SEARCH.ticket_search_multi_index;
DROP TABLE IF EXISTS SE_FEATURES.CORTEX_SEARCH.support_tickets;
DROP SCHEMA IF EXISTS SE_FEATURES.CORTEX_SEARCH;
DROP DATABASE IF EXISTS cortex_search_demo;
DROP WAREHOUSE IF EXISTS cortex_search_wh;